# 🦀 Day 4: FFI — Exposing Rust to C
---

Today we learn how to make **Rust callable from C**.

- **`#[no_mangle]`**: Prevent name mangling so C can find the symbol
- **`extern "C" fn`**: C-compatible function signatures
- **`crate-type`**: `cdylib` or `staticlib` for C consumption
- **Return values**: Primitives, pointers; string handling with `*const c_char`
- **Memory management**: `Box::into_raw` and `Box::from_raw`
- **cbindgen**: Auto-generate C headers from Rust

In [ ]:
// Making Rust callable from C: #[no_mangle] + extern "C"
use std::ffi::{c_int, c_char, CString};

#[no_mangle]
pub extern "C" fn rust_add(a: c_int, b: c_int) -> c_int {
    a + b
}

#[no_mangle]
pub extern "C" fn rust_greet(name: *const c_char) -> *mut c_char {
    // Use CStr::from_ptr(name) to read input; CString::new for return
    // Caller must free the returned pointer (CString::from_raw)
    let greeting = CString::new("Hello from Rust!").unwrap();
    greeting.into_raw()
}

// Call our own exported function (simulating C calling us)
let sum = rust_add(10, 20);
println!("rust_add(10, 20) = {}", sum);

In [ ]:
// Box::into_raw and Box::from_raw for heap allocations across FFI
// into_raw: Box<T> -> *mut T (caller owns, must free)
// from_raw: *mut T -> Box<T> (Rust takes ownership back)

fn create_box() -> *mut i32 {
    Box::into_raw(Box::new(42))
}

fn destroy_box(ptr: *mut i32) {
    if !ptr.is_null() {
        unsafe { drop(Box::from_raw(ptr)); }
    }
}

let ptr = create_box();
unsafe { println!("Value: {}", *ptr); }
destroy_box(ptr);

## Cargo project: Building a C-exported library

In `Cargo.toml`:
```toml
[lib]
crate-type = ["cdylib"]  # dynamic library (.so, .dll)
# or crate-type = ["staticlib"]  # static library (.a, .lib)
```

Then `cargo build` produces `libyourlib.so` (Unix) or `yourlib.dll` (Windows).

**cbindgen** generates C headers:
```toml
[build-dependencies]
cbindgen = "0.26"
```

In `build.rs`:
```rust
fn main() {
    cbindgen::generate(".").expect("cbindgen failed")
        .write_to_file("target/yourlib.h");
}
```

## 📝 Exercise: Design a C API for a Rust key-value store

Design the C function signatures for:
- `kv_store* kv_create()`
- `void kv_set(kv_store* s, const char* key, const char* value)`
- `char* kv_get(kv_store* s, const char* key)`  // caller frees
- `void kv_destroy(kv_store* s)`

Write the `extern "C"` declarations and stub implementations. Consider: who owns memory? Who frees?

In [ ]:
// YOUR CODE HERE

## 🎯 Key Takeaways

1. **`#[no_mangle]`** keeps symbol names as-is so C can link
2. **`extern "C" fn`** uses C calling convention
3. **`crate-type = ["cdylib"]`** or **`["staticlib"]`** for C consumption
4. **`Box::into_raw`** / **`Box::from_raw`** transfer heap ownership across FFI
5. **Document ownership** — who allocates, who frees — in your C API
6. **cbindgen** auto-generates `.h` files from Rust

---
**Tomorrow:** bindgen — generating Rust bindings from C headers